<a href="https://colab.research.google.com/github/SolomonShilly/Gymnasium-RL/blob/main/CartPole_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
"""
THE CORE RL LOOP
==========================
Environment - the world the agent interacts with
              (e.g., a game board, a physical simulation)

State       - a snapshot of the environment at a given moment

Agent       - the learner/decision-maker

Action      - a choice the agent makes from a set of possible options,
              given the current state

Reward      - a number the environment gives back after each action,
              telling the agent how good or bad that action was.
              This is the most important part of RL - it's what
              distinguishes RL from ordinary supervised learning.

Policy      - the agent's strategy for choosing actions based on states;
              this is literally what the agent is "learning" over time

Episode     - one full run from a starting state to an ending state
              (e.g., one full game)

Loop: state -> action -> reward -> new state -> repeat
"""

'\nTHE CORE RL LOOP\n==========================\nEnvironment - the world the agent interacts with\n              (e.g., a game board, a physical simulation)\n\nState       - a snapshot of the environment at a given moment\n\nAgent       - the learner/decision-maker\n\nAction      - a choice the agent makes from a set of possible options,\n              given the current state\n\nReward      - a number the environment gives back after each action,\n              telling the agent how good or bad that action was.\n              This is the most important part of RL - it\'s what\n              distinguishes RL from ordinary supervised learning.\n\nPolicy      - the agent\'s strategy for choosing actions based on states;\n              this is literally what the agent is "learning" over time\n\nEpisode     - one full run from a starting state to an ending state\n              (e.g., one full game)\n\nLoop: state -> action -> reward -> new state -> repeat\n'

In [16]:
# The ! at the start tells Colab to run it as a shell command instead of Python code
!pip install gymnasium

In [17]:
import gymnasium as gym
gym.envs.registry.keys()

dict_keys(['CartPole-v0', 'CartPole-v1', 'MountainCar-v0', 'MountainCarContinuous-v0', 'Pendulum-v1', 'Acrobot-v1', 'phys2d/CartPole-v0', 'phys2d/CartPole-v1', 'phys2d/Pendulum-v0', 'LunarLander-v3', 'LunarLanderContinuous-v3', 'BipedalWalker-v3', 'BipedalWalkerHardcore-v3', 'CarRacing-v3', 'Blackjack-v1', 'FrozenLake-v1', 'FrozenLake8x8-v1', 'CliffWalking-v1', 'CliffWalkingSlippery-v1', 'Taxi-v4', 'tabular/Blackjack-v0', 'tabular/CliffWalking-v0', 'Reacher-v2', 'Reacher-v4', 'Reacher-v5', 'Pusher-v2', 'Pusher-v4', 'Pusher-v5', 'InvertedPendulum-v2', 'InvertedPendulum-v4', 'InvertedPendulum-v5', 'InvertedDoublePendulum-v2', 'InvertedDoublePendulum-v4', 'InvertedDoublePendulum-v5', 'HalfCheetah-v2', 'HalfCheetah-v3', 'HalfCheetah-v4', 'HalfCheetah-v5', 'Hopper-v2', 'Hopper-v3', 'Hopper-v4', 'Hopper-v5', 'Swimmer-v2', 'Swimmer-v3', 'Swimmer-v4', 'Swimmer-v5', 'Walker2d-v2', 'Walker2d-v3', 'Walker2d-v4', 'Walker2d-v5', 'Ant-v2', 'Ant-v3', 'Ant-v4', 'Ant-v5', 'Humanoid-v2', 'Humanoid-v3', 

In [18]:
env = gym.make("CartPole-v1") # Creating an environment from the gymnasium registry to simulate a pole balancing on a moving cart
state, info = env.reset() # Reset to change starting state to practice generalization and not memorization
print("Initial state:", state)

"""
State Values Represents:
Cart position (how far left/right it is on the track)
Cart velocity (how fast it's moving)
Pole angle (how tilted the pole is)
Pole angular velocity (how fast the pole is tilting)
"""

Initial state: [-0.04430786 -0.02693972 -0.03617227  0.02471451]


"\nState Values Represents:\nCart position (how far left/right it is on the track)\nCart velocity (how fast it's moving)\nPole angle (how tilted the pole is)\nPole angular velocity (how fast the pole is tilting)\n"

In [19]:
action = env.action_space.sample() # Only two possible actions for the cart are to move left or right
next_state, reward, terminated, truncated, info = env.step(action) # When you apply the action, the built-in physics simulation will calculate what happens next
print("Action taken:", action)
print("Next State: ", next_state)
print("Reward received", reward) # +1 for every single timestep the pole stays upright (doesn't fall past a certain angle) and the cart stays on the track (doesn't go too far left/right).

Action taken: 0
Next State:  [-0.04484666 -0.22152476 -0.03567798  0.30576882]
Reward received 1.0


In [20]:
""" Q-Value: the average score of an action
new_avg = [old_avg × (n-1) + new_result] / n

Step 1 — split the fraction into two pieces (distribute the division by n across the addition):

new_avg = [old_avg × (n-1)] / n  +  new_result / n

Step 2 — expand (n-1)/n as n/n - 1/n, which is just 1 - 1/n:

new_avg = old_avg × (1 - 1/n)  +  new_result / n

Step 3 — distribute old_avg across that:

new_avg = old_avg - old_avg/n + new_result/n

Step 4 — group the two fractions together:

new_avg = old_avg + (new_result - old_avg)/n

Q-values are the running average of each possible action, so the agent can educate itself on the environment.
The Q-table is a collection of all possible combinations of all state values based on the possible buckets available.
We can forecast the number of rows and columns in the Q-table by multiplying buckets by itself for every state value and multiplying that product by the possible actions

Early on agents should update fast and change their mind easily, but as n grows, they update slow and trust their experience
The curse of Dimensionality: when you bucket multiple state values, the possible unique situations grows rapidly. This is why neural networks are preferred over RL
"""

' Q-Value: the average score of an action\nnew_avg = [old_avg × (n-1) + new_result] / n\n\nStep 1 — split the fraction into two pieces (distribute the division by n across the addition):\n\nnew_avg = [old_avg × (n-1)] / n  +  new_result / n\n\nStep 2 — expand (n-1)/n as n/n - 1/n, which is just 1 - 1/n:\n\nnew_avg = old_avg × (1 - 1/n)  +  new_result / n\n\nStep 3 — distribute old_avg across that:\n\nnew_avg = old_avg - old_avg/n + new_result/n\n\nStep 4 — group the two fractions together:\n\nnew_avg = old_avg + (new_result - old_avg)/n\n\nQ-values are the running average of each possible action, so the agent can educate itself on the environment.\nThe Q-table is a collection of all possible combinations of all state values based on the possible buckets available.\nWe can forecast the number of rows and columns in the Q-table by multiplying buckets by itself for every state value and multiplying that product by the possible actions\n\nEarly on agents should update fast and change their

In [21]:
import numpy as np

def get_bucket(value, low, high, num_buckets):
  value = np.clip(value, low, high) # Clipping value, so it does not go outside the range of low and high angles for the pole
  ratio = (value - low) / (high - low) # Calculating how far across the range the value is
  bucket = int(ratio * num_buckets) # Assigning each value to a bucket using the ratio, we use int to truncate to avoid rounding
  bucket = min(bucket, num_buckets - 1) # If the ratio is 1, we assign the value to the last bucket which is n-1 since we start counting with 0
  return bucket

get_bucket(0.0293, -0.2, 0.2, 3)

1

In [22]:
# Each of the 4 state values needs its own low and high, because they don't share a range.
state = [-0.04516448, 0.14973801, 0.02858541, -0.3176672]

cart_pos_bucket = get_bucket(state[0], -2.4, 2.4, 3)
cart_vel_bucket = get_bucket(state[1], -3.0, 3.0, 3)
pole_angle_bucket = get_bucket(state[2], -0.2, 0.2, 3)
pole_vel_bucket = get_bucket(state[3], -3.5, 3.5, 3)

print("Cart position bucket:", cart_pos_bucket)
print("Cart velocity bucket:", cart_vel_bucket)
print("Pole angle bucket:", pole_angle_bucket)
print("Pole velocity bucket:", pole_vel_bucket)

num_buckets = 6
num_actions = 2

Q = np.zeros((num_buckets, num_buckets, num_buckets, num_buckets, num_actions)) # Creating 5D array of zeros which is our Q-table


Cart position bucket: 1
Cart velocity bucket: 1
Pole angle bucket: 1
Pole velocity bucket: 1


In [23]:
def choose_action(state_buckets, Q, epsilon, num_actions):
    if np.random.random() < epsilon:
        return np.random.randint(num_actions)  # explore: random action
    else:
        return np.argmax(Q[state_buckets])  # exploit: best known action

state_buckets = (cart_pos_bucket, cart_vel_bucket, pole_angle_bucket, pole_vel_bucket)
action = choose_action(state_buckets, Q, epsilon=0.1, num_actions=2)
print("Chosen action:", action)

Chosen action: 0


In [24]:
def update_q(Q, state_buckets, action, reward, next_state_buckets, alpha, gamma):
    best_next_value = np.max(Q[next_state_buckets])
    td_target = reward + gamma * best_next_value
    td_error = td_target - Q[state_buckets][action]
    Q[state_buckets][action] += alpha * td_error

current_state_buckets = (cart_pos_bucket, cart_vel_bucket, pole_angle_bucket, pole_vel_bucket)
action = 0
reward = 1.0
next_state = [-0.04516448, 0.14973801, 0.02858541, -0.3176672]  # example next state
next_state_buckets = (
    get_bucket(next_state[0], -2.4, 2.4, num_buckets),
    get_bucket(next_state[1], -3.0, 3.0, num_buckets),
    get_bucket(next_state[2], -0.2, 0.2, num_buckets),
    get_bucket(next_state[3], -3.5, 3.5, num_buckets),
)

update_q(Q, current_state_buckets, action, reward, next_state_buckets, alpha=0.1, gamma=0.9)
print(Q[current_state_buckets])

[0.1 0. ]


In [26]:
num_episodes = 20000
alpha = 0.1
gamma = 0.999
epsilon = 1.0
epsilon_min = 0.01
epsilon_decay = 0.999

bounds = [(-2.4, 2.4), (-3.0, 3.0), (-0.2, 0.2), (-3.5, 3.5)]

def get_state_buckets(state, bounds, num_buckets):
    return tuple(
        get_bucket(state[i], bounds[i][0], bounds[i][1], num_buckets)
        for i in range(4)
    )

for episode in range(num_episodes):
    state, info = env.reset()
    state_buckets = get_state_buckets(state, bounds, num_buckets)
    done = False
    total_reward = 0

    while not done:
        action = choose_action(state_buckets, Q, epsilon, num_actions)
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        next_state_buckets = get_state_buckets(next_state, bounds, num_buckets)

        update_q(Q, state_buckets, action, reward, next_state_buckets, alpha, gamma)

        state_buckets = next_state_buckets
        total_reward += reward

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    if episode % 500 == 0:
        print(f"Episode {episode}, total reward: {total_reward}, epsilon: {epsilon:.3f}")

Episode 0, total reward: 15.0, epsilon: 0.999
Episode 500, total reward: 18.0, epsilon: 0.606
Episode 1000, total reward: 13.0, epsilon: 0.367
Episode 1500, total reward: 14.0, epsilon: 0.223
Episode 2000, total reward: 17.0, epsilon: 0.135
Episode 2500, total reward: 12.0, epsilon: 0.082
Episode 3000, total reward: 17.0, epsilon: 0.050
Episode 3500, total reward: 13.0, epsilon: 0.030
Episode 4000, total reward: 12.0, epsilon: 0.018
Episode 4500, total reward: 37.0, epsilon: 0.011
Episode 5000, total reward: 11.0, epsilon: 0.010
Episode 5500, total reward: 10.0, epsilon: 0.010
Episode 6000, total reward: 13.0, epsilon: 0.010
Episode 6500, total reward: 12.0, epsilon: 0.010
Episode 7000, total reward: 12.0, epsilon: 0.010
Episode 7500, total reward: 11.0, epsilon: 0.010
Episode 8000, total reward: 13.0, epsilon: 0.010
Episode 8500, total reward: 13.0, epsilon: 0.010
Episode 9000, total reward: 13.0, epsilon: 0.010
Episode 9500, total reward: 39.0, epsilon: 0.010
Episode 10000, total rew